In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("database/ecommerce.db")

print("Database Connected Successfully")

Database Connected Successfully


In [3]:
query = """
select p.category,
    ROUND( SUM(oi.quantity * oi.unit_price *
            (1 - oi.discount_percent/100)
        ),2
    ) AS total_revenue
from order_items oi
JOIN products p
ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

pd.read_sql(query, conn)

,category,total_revenue
0,Clothing,955736.26
1,Electronics,932594.05
2,Books,796833.42
3,Home,769129.77


In [18]:
query = """
select o.customer_id,
    ROUND( SUM( oi.quantity * oi.unit_price *
            (1-oi.discount_percent/100)
        ),2
    ) AS total_spent
from orders o

JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY o.customer_id
ORDER BY total_spent DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_id,total_spent
0,UNKNOWN,152517.06
1,C00899,20296.34
2,C00469,17118.14
3,C00476,15977.97
4,C00567,15140.30
5,C00490,14670.57
6,C00820,14573.33
7,C00181,14066.27
8,C00188,13862.00
9,C00184,13513.40


In [17]:
query = """
select
    strftime('%Y-%m',order_date) AS month,
    COUNT(order_id) AS total_orders
from orders
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query, conn)

,month,total_orders
0,2024-07,76
1,2024-08,70
2,2024-09,100
3,2024-10,112
4,2024-11,74
5,2024-12,87
6,2025-01,85
7,2025-02,77
8,2025-03,80
9,2025-04,76


In [19]:
query = """
select DISTINCT customer_id
from orders
WHERE customer_id NOT IN (
                        SELECT customer_id
                        FROM orders
                        WHERE status='DELIVERED'
                        );
"""

pd.read_sql(query, conn)

,customer_id
0,C00657
1,C00565
2,C00880
3,C00752
4,C00788
...,...
508,C00738
509,C00477
510,C00239
511,C00512


In [20]:
query = """
select product_id,
    SUM( CASE
    WHEN quantity<0 THEN 1
    ELSE 0 END ) AS returns,
SUM( CASE
WHEN quantity>0 THEN 1
ELSE 0 END ) AS purchases

from order_items
GROUP BY product_id
HAVING returns > purchases;
"""

pd.read_sql(query, conn)

,product_id,returns,purchases


In [ ]:
query = """
select p.category, ROUND( 100.0* 
    SUM(CASE WHEN oi.quantity<0 THEN 1 ELSE 0 END)/COUNT(*) ,2) AS return_rate

from order_items oi
JOIN products p
ON oi.product_id=p.product_id
GROUP BY p.category;
"""

pd.read_sql(query, conn)

,category,return_rate
0,Books,3.66
1,Clothing,3.90
2,Electronics,3.31
3,Home,2.41


In [21]:
query = """
WITH daily AS ( SELECT o.region_code, DATE(o.order_date) AS order_day,
    SUM(oi.quantity * oi.unit_price* 
    (1-oi.discount_percent/100)
    ) AS daily_revenue

from orders o
JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY o.region_code,order_day )

select region_code, order_day,
    ROUND(daily_revenue,2),
    ROUND( SUM(daily_revenue) 
    OVER( PARTITION BY region_code ORDER BY order_day) ,2)
    AS running_total
from daily;
"""

pd.read_sql(query, conn)

,region_code,order_day,"ROUND(daily_revenue,2)",running_total
0,East,2024-07-08,2052.52,2052.52
1,East,2024-07-10,3760.04,5812.56
2,East,2024-07-11,4719.08,10531.65
3,East,2024-07-13,3035.58,13567.23
4,East,2024-07-15,842.29,14409.52
...,...,...,...,...
1422,West,2026-07-03,43.65,857145.88
1423,West,2026-07-04,2528.20,859674.08
1424,West,2026-07-05,2495.74,862169.82
1425,West,2026-07-06,78.00,862247.82


In [ ]:
query = """
SELECT * FROM(SELECT p.category, p.product_name,
    ROUND( SUM(oi.quantity* oi.unit_price*(1-oi.discount_percent/100)),2) AS revenue,
    DENSE_RANK() OVER( PARTITION BY p.category
ORDER BY SUM( oi.quantity* oi.unit_price* (1-oi.discount_percent/100)) DESC) AS rank_in_category
FROM order_items oi
JOIN products p
ON oi.product_id=p.product_id
GROUP BY p.category, p.product_name )
ORDER BY category,
rank_in_category;
"""

pd.read_sql(query, conn)

,category,product_name,revenue,rank_in_category
0,Books,Us Comics,16095.21,1
1,Books,Edge Fiction,16015.98,2
2,Books,Anything Education,15165.99,3
3,Books,Red Comics,14996.92,4
4,Books,List Education,13015.59,5
...,...,...,...,...
490,Home,Event Furniture,2397.44,106
491,Home,Improve Kitchen,2301.64,107
492,Home,Week Kitchen,1352.83,108
493,Home,Near Furniture,469.98,109


In [ ]:
query = """
SELECT customer_id, order_date,
LAG(order_date)
    OVER(PARTITION BY customer_id ORDER BY order_date ) AS previous_order,
    JULIANDAY(order_date)- ULIANDAY(LAG(order_date)
    OVER( PARTITION BY customer_id
ORDER BY order_date)
)
AS days_gap

FROM orders;
"""

pd.read_sql(query, conn)

,customer_id,order_date,previous_order,days_gap
0,C00001,2024-10-13 23:32:28,None,NaN
1,C00004,2025-02-25 05:37:51,None,NaN
2,C00004,2025-04-15 05:21:42,2025-02-25 05:37:51,48.988785
3,C00005,2024-07-31 19:30:23,None,NaN
4,C00005,2025-05-30 15:20:34,2024-07-31 19:30:23,302.826516
...,...,...,...,...
1995,UNKNOWN,2026-06-07 19:34:30,2026-05-29 07:03:54,9.521250
1996,UNKNOWN,2026-06-20 21:12:48,2026-06-07 19:34:30,13.068264
1997,UNKNOWN,2026-06-26 07:24:59,2026-06-20 21:12:48,5.425127
1998,UNKNOWN,2026-07-03 16:56:37,2026-06-26 07:24:59,7.396968


In [ ]:
query = """
WITH revenue AS( SELECT o.customer_id,
    SUM( oi.quantity* oi.unit_price* (1-oi.discount_percent/100)) AS revenue
FROM orders o
JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY o.customer_id
)

SELECT
customer_id, ROUND(revenue,2),
CASE
    WHEN revenue>10000 THEN 'High'
    WHEN revenue>=5000 THEN 'Medium'
ELSE 'Low'
END AS segment
FROM revenue
ORDER BY revenue DESC;
"""

pd.read_sql(query, conn)

,customer_id,"ROUND(revenue,2)",segment
0,UNKNOWN,152517.06,High
1,C00899,20296.34,High
2,C00469,17118.14,High
3,C00476,15977.97,High
4,C00567,15140.30,High
...,...,...,...
846,C00598,-1117.53,Low
847,C00099,-1383.97,Low
848,C00142,-1444.25,Low
849,C00817,-1668.17,Low


In [ ]:
query = """
WITH revenue AS(SELECT o.customer_id,
    SUM( oi.quantity* oi.unit_price * (1-oi.discount_percent/100)) AS revenue
    FROM orders o
    JOIN order_items oi
    ON o.order_id=oi.order_id
    GROUP BY o.customer_id )
SELECT customer_id,
    ROUND(revenue,2), NTILE(4) OVER(ORDER BY revenue DESC ) AS quartile
FROM revenue;
"""

pd.read_sql(query, conn)

,customer_id,"ROUND(revenue,2)",quartile
0,UNKNOWN,152517.06,1
1,C00899,20296.34,1
2,C00469,17118.14,1
3,C00476,15977.97,1
4,C00567,15140.30,1
...,...,...,...
846,C00598,-1117.53,4
847,C00099,-1383.97,4
848,C00142,-1444.25,4
849,C00817,-1668.17,4


In [ ]:
query = """
WITH purchase_rank AS (SELECT o.customer_id, p.category, o.order_date,
    ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date ) AS rn
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id

JOIN products p 
ON oi.product_id = p.product_id )
SELECT customer_id, category AS first_category
FROM purchase_rank
WHERE rn = 1;
"""

pd.read_sql(query, conn)

,customer_id,first_category
0,C00001,Clothing
1,C00004,Home
2,C00005,Home
3,C00006,Clothing
4,C00007,Clothing
...,...,...
846,C00997,Books
847,C00998,Electronics
848,C00999,Clothing
849,C01000,Electronics


In [ ]:
query = """
select
    a.product_id AS product_a, 
    b.product_id AS product_b,
    COUNT(*) AS times_bought_together
from order_items a
JOIN order_items b
ON a.order_id=b.order_id
AND a.product_id<b.product_id
GROUP BY product_a, product_b
ORDER BY times_bought_together DESC
LIMIT 20;
"""

pd.read_sql(query, conn)

,product_a,product_b,times_bought_together
0,P0002,P0142,3
1,P0161,P0374,3
2,P0295,P0429,3
3,P0002,P0416,2
4,P0003,P0419,2
5,P0007,P0378,2
6,P0014,P0378,2
7,P0016,P0072,2
8,P0016,P0202,2
9,P0017,P0019,2


In [ ]:
query = """
SELECT p.product_name,
    ROUND(
    SUM(oi.quantity * oi.unit_price*
    (1-oi.discount_percent/100)),2) AS revenue
FROM products p
JOIN order_items oi
ON p.product_id=oi.product_id
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_name,revenue
0,Join Kids,21138.23
1,Decade Laptop,20060.04
2,Month Men,19594.19
3,Follow Men,18612.96
4,Eight Mobile,18129.54
5,Bill Mobile,17981.37
6,Friend Laptop,16944.80
7,Continue Decor,16320.55
8,Yard Decor,16200.87
9,Here Mobile,16112.37


In [4]:
query = """
WITH MonthlyRev AS (
    SELECT 
        strftime('%Y', order_date) AS year,
        strftime('%m', order_date) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY year, month
)
SELECT 
    curr.year,
    curr.month,
    ROUND(curr.revenue, 2) AS revenue,
    ROUND(prev.revenue, 2) AS prev_year_revenue,
    ROUND(((curr.revenue - prev.revenue) / prev.revenue) * 100, 2) AS yoy_growth_percent
FROM MonthlyRev curr
LEFT JOIN MonthlyRev prev 
    ON curr.month = prev.month 
    AND CAST(curr.year AS INTEGER) = CAST(prev.year AS INTEGER) + 1
ORDER BY curr.year, curr.month;
"""
pd.read_sql(query, conn)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,07,143217.92,NaN,NaN
1,2024,08,121638.68,NaN,NaN
2,2024,09,179206.46,NaN,NaN
3,2024,10,193375.21,NaN,NaN
4,2024,11,142468.25,NaN,NaN
5,2024,12,129544.26,NaN,NaN
6,2025,01,138483.00,NaN,NaN
7,2025,02,125423.16,NaN,NaN
8,2025,03,142307.82,NaN,NaN
9,2025,04,142672.88,NaN,NaN


In [5]:
query = """
WITH Cohorts AS (
    SELECT customer_id, strftime('%Y-%m', registration_date) AS cohort_month
    FROM customers
),
OrderMonths AS (
    SELECT 
        c.customer_id,
        c.cohort_month,
        CAST(strftime('%Y', o.order_date) AS INTEGER) * 12 + CAST(strftime('%m', o.order_date) AS INTEGER) -
        (CAST(substr(c.cohort_month, 1, 4) AS INTEGER) * 12 + CAST(substr(c.cohort_month, 6, 2) AS INTEGER)) AS month_diff
    FROM Cohorts c
    JOIN orders o ON c.customer_id = o.customer_id
),
CohortSize AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS total_customers
    FROM Cohorts
    GROUP BY cohort_month
),
Retention AS (
    SELECT cohort_month, month_diff, COUNT(DISTINCT customer_id) AS active_customers
    FROM OrderMonths
    WHERE month_diff IN (0, 1, 2, 3)
    GROUP BY cohort_month, month_diff
)
SELECT 
    r.cohort_month,
    cs.total_customers,
    SUM(CASE WHEN r.month_diff = 0 THEN r.active_customers ELSE 0 END) AS month_0_active,
    SUM(CASE WHEN r.month_diff = 1 THEN r.active_customers ELSE 0 END) AS month_1_active,
    SUM(CASE WHEN r.month_diff = 2 THEN r.active_customers ELSE 0 END) AS month_2_active,
    SUM(CASE WHEN r.month_diff = 3 THEN r.active_customers ELSE 0 END) AS month_3_active,
    ROUND(100.0 * SUM(CASE WHEN r.month_diff = 1 THEN r.active_customers ELSE 0 END) / cs.total_customers, 2) AS month_1_retention_pct
FROM Retention r
JOIN CohortSize cs ON r.cohort_month = cs.cohort_month
GROUP BY r.cohort_month
ORDER BY r.cohort_month;
"""
pd.read_sql(query, conn)

,cohort_month,total_customers,month_0_active,month_1_active,month_2_active,month_3_active,month_1_retention_pct
0,2024-05,37,0,0,1,0,0.00
1,2024-06,26,0,3,1,7,11.54
2,2024-07,21,1,2,2,1,9.52
3,2024-08,26,1,1,3,1,3.85
4,2024-09,35,1,3,2,2,8.57
5,2024-10,26,3,3,3,2,11.54
6,2024-11,26,2,3,3,1,11.54
7,2024-12,25,1,1,0,3,4.00
8,2025-01,27,4,0,3,0,0.00
9,2025-02,25,0,1,4,4,4.00


In [ ]:
conn.close()

print("Notebook 3 Completed Successfully")